In [1]:
using Piccolo
using CairoMakie
using LinearAlgebra
using SparseArrays
using QuantumOpticsBase

In [6]:
const Bₐ = SpinBasis(1//2)
const Bₚ = FockBasis(5)

const g = spindown(Bₐ)
const e = spinup(Bₐ)

const f₀ = fockstate(Bₚ, 0)
const f₁ = fockstate(Bₚ, 1)
const f₂ = fockstate(Bₚ, 2)
const f₄ = fockstate(Bₚ, 4)
const f₅ = fockstate(Bₚ, 5)

const σ₋ = sigmam(Bₐ)
const σ₊ = sigmap(Bₐ)
const σx = sigmax(Bₐ)
const σz = sigmaz(Bₐ)
const σy = sigmay(Bₐ)
const a = destroy(Bₚ)
const n = number(Bₚ)

const Iₐ = identityoperator(Bₐ)
const Iₚ = identityoperator(Bₚ)

const Hₐ = projector(e) ⊗ Iₚ
const Hₚ = Iₐ ⊗ n

σx

Operator(dim=2x2)
  basis: Spin(1/2)
      ⋅       1.0 + 0.0im
 1.0 + 0.0im       ⋅     

In [ ]:
# make dense matrix
using LinearAlgebra
using SparseArrays
H1 =  full((σx⊗Iₚ).data)

# system = QuantumSystem(
#     H_drift = σz⊗n,
#     H_drives = [
#         (σx⊗Iₚ).data,
# 		(σy⊗Iₚ).data,
# 		(Iₐ⊗(a+a')).data,
# 		(im*Iₐ⊗(a-a')).data
#     ]
# )

UndefVarError: UndefVarError: `full` not defined in `Main`
Suggestion: check for spelling errors or missing imports.
Hint: a global variable of this name may be made accessible by importing PDMats in the current active module Main

In [3]:

function UnitarySmoothPulseProblem(
    system::AbstractQuantumSystem,
    goal::AbstractPiccoloOperator,
    T::Int,
    Δt::Union{Float64, <:AbstractVector{Float64}};
    unitary_integrator=UnitaryIntegrator,
    state_name::Symbol = :Ũ⃗,
    control_name::Symbol = :a,
    timestep_name::Symbol = :Δt,
    init_trajectory::Union{NamedTrajectory, Nothing}=nothing,
    a_guess::Union{Matrix{Float64}, Nothing}=nothing,
    a_bound::Float64=1.0,
    a_bounds=fill(a_bound, system.n_drives),
    da_bound::Float64=Inf,
    da_bounds=fill(da_bound, system.n_drives),
    dda_bound::Float64=1.0,
    dda_bounds=fill(dda_bound, system.n_drives),
    Δt_min::Float64=0.5 * minimum(Δt),
    Δt_max::Float64=2.0 * maximum(Δt),
    Q::Float64=100.0,
    R=1e-2,
    R_a::Union{Float64, Vector{Float64}}=R,
    R_da::Union{Float64, Vector{Float64}}=R,
    R_dda::Union{Float64, Vector{Float64}}=R,
    constraints::Vector{<:AbstractConstraint}=AbstractConstraint[],
    piccolo_options::PiccoloOptions=PiccoloOptions(),
)
    if piccolo_options.verbose
        println("    constructing UnitarySmoothPulseProblem...")
        println("\tusing integrator: $(typeof(unitary_integrator))")
    end

    # Trajectory
    if !isnothing(init_trajectory)
        traj = init_trajectory
    else
        traj = initialize_trajectory(
            goal,
            T,
            Δt,
            system.n_drives,
            (a_bounds, da_bounds, dda_bounds);
            state_name=state_name,
            control_name=control_name,
            timestep_name=timestep_name,
            Δt_bounds=(Δt_min, Δt_max),
            zero_initial_and_final_derivative=piccolo_options.zero_initial_and_final_derivative,
            geodesic=piccolo_options.geodesic,
            bound_state=piccolo_options.bound_state,
            a_guess=a_guess,
            system=system,
            rollout_integrator=piccolo_options.rollout_integrator,
            verbose=piccolo_options.verbose
        )
    end

    # Objective
    J = UnitaryInfidelityObjective(goal, state_name, traj; Q=Q)

    control_names = [
        name for name ∈ traj.names
            if endswith(string(name), string(control_name))
    ]

    J += QuadraticRegularizer(control_names[1], traj, R_a)
    J += QuadraticRegularizer(control_names[2], traj, R_da)
    J += QuadraticRegularizer(control_names[3], traj, R_dda)

    # Optional Piccolo constraints and objectives
    J += apply_piccolo_options!(
        piccolo_options, constraints, traj; 
        state_names=state_name,
        state_leakage_indices=goal isa EmbeddedOperator ? 
            get_iso_vec_leakage_indices(goal) : 
            nothing
    )

    integrators = [
        unitary_integrator(system, traj, state_name, control_name),
        DerivativeIntegrator(traj, control_name, control_names[2]),
        DerivativeIntegrator(traj, control_names[2], control_names[3]),
    ]

    return DirectTrajOptProblem(
        traj,
        J,
        integrators;
        constraints=constraints
    )
end

function UnitarySmoothPulseProblem(
    H_drift::AbstractMatrix{<:Number},
    H_drives::Vector{<:AbstractMatrix{<:Number}},
    args...;
    kwargs...
)
    system = QuantumSystem(H_drift, H_drives)
    return UnitarySmoothPulseProblem(system, args...; kwargs...)
end


UnitarySmoothPulseProblem (generic function with 2 methods)

In [ ]:
system = QuantumSystem(
    H_drift = [0.0 1.0; 1.0 0.0],
    H_drives = [1.0 * [1.0 0.0; 0.0 1.0], 1.0 * [0.0 1.0; 1.0 0.0]]
)